In [18]:
import os
import httpx
from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma  
from langchain_core.prompts import PromptTemplate 
load_dotenv()

True

In [2]:
dummy_transcript = {
    "lang": "en",
    "availableLangs": ["en"],
    "content": [
        {
            "lang": "en",
            "text": "Welcome everyone. Today we're going to talk about machine learning fundamentals.",
            "offset": 0,
            "duration": 5200
        },
        {
            "lang": "en",
            "text": "Many beginners jump straight into deep learning without understanding the basics.",
            "offset": 5200,
            "duration": 4800
        },
        {
            "lang": "en",
            "text": "You should first understand linear regression, classification, and evaluation metrics.",
            "offset": 10000,
            "duration": 6100
        },
        {
            "lang": "en",
            "text": "Once you're comfortable with those concepts, neural networks become much easier to understand.",
            "offset": 16100,
            "duration": 6700
        },
        {
            "lang": "en",
            "text": "Let's now build our first model using Python and scikit-learn.",
            "offset": 22800,
            "duration": 5400
        },
        {
            "lang": "en",
            "text": "We'll split our dataset into training and testing sets before fitting the model.",
            "offset": 28200,
            "duration": 6200
        },
        {
            "lang": "en",
            "text": "Finally, we'll evaluate the model using accuracy, precision, recall, and F1 score.",
            "offset": 34400,
            "duration": 7000
        },
        {
            "lang": "en",
            "text": "In the next lecture we'll explore decision trees and random forests in more detail.",
            "offset": 41400,
            "duration": 6300
        }
    ]
}

In [3]:
def get_transcript(video_id: str) -> dict: #key is content  and content contains a list of dictionaries with keys as lang text offset duration
    # SUPADATA_API_KEY = os.environ.get("SUPADATA_API_KEY")
    
    # response = httpx.get(
    #     "https://api.supadata.ai/v1/youtube/transcript",
    #     params={"videoId": video_id},
    #     headers={"x-api-key": SUPADATA_API_KEY},
    #     timeout=60.0,
    # )
    
    # if response.status_code != 200:
    #     raise Exception(f"Supadata error: {response.status_code} - {response.text}")
    
    # data = response.json()
    
    # return data
    return dummy_transcript

data = get_transcript("WDSRXu4cJbM")

In [ ]:
docs = []
current_text = ""
start_offset = None

for segment in data["content"]:
    if start_offset is None:
        start_offset = segment["offset"]

    current_text += " " + segment["text"]

    # create a chunk every 1000 characters
    if len(current_text) >= 100:
        docs.append(
            Document(
                page_content=current_text.strip(),
                metadata={"offset": start_offset}
            )
        )
        current_text = ""
        start_offset = None

# remaining text
if current_text:
    docs.append(
        Document(
            page_content=current_text.strip(),
            metadata={"offset": start_offset}
        )
    )

docs

[Document(metadata={'offset': 0}, page_content="Welcome everyone. Today we're going to talk about machine learning fundamentals. Many beginners jump straight into deep learning without understanding the basics."),
 Document(metadata={'offset': 10000}, page_content="You should first understand linear regression, classification, and evaluation metrics. Once you're comfortable with those concepts, neural networks become much easier to understand."),
 Document(metadata={'offset': 22800}, page_content="Let's now build our first model using Python and scikit-learn. We'll split our dataset into training and testing sets before fitting the model."),
 Document(metadata={'offset': 34400}, page_content="Finally, we'll evaluate the model using accuracy, precision, recall, and F1 score. In the next lecture we'll explore decision trees and random forests in more detail.")]

In [5]:
vector_store = Chroma(
    embedding_function=OpenAIEmbeddings(
        model="openai/text-embedding-3-large",
        dimensions=1536,
        api_key=os.getenv("OPENROUTER_API_KEY"),
        base_url="https://openrouter.ai/api/v1"
    ),
    persist_directory='my_chroma_db',
    collection_name='videos_transcript_chunks_1536'
)


vector_store.add_documents(docs)

['4b0d4138-3ac0-4c20-96f4-08609c2ecb91',
 'b92038de-b550-4612-9926-91b5187128fb',
 'dfb5235d-468d-44f1-b9bf-01d7e0c01061',
 '67e720d9-3c2b-458f-8bb0-72260b220c8e']

In [6]:
vector_store.similarity_search(
    query="where did he talk about linear regression and classification",
    k=1
)

[Document(id='b92038de-b550-4612-9926-91b5187128fb', metadata={'offset': 10000}, page_content="You should first understand linear regression, classification, and evaluation metrics. Once you're comfortable with those concepts, neural networks become much easier to understand.")]

In [11]:
retriever = vector_store.as_retriever(search_kwargs={"k" : 1})

In [ ]:
# query = "where did he talk about precision and recall"
# result = retriever.invoke(query)
# result

[Document(id='67e720d9-3c2b-458f-8bb0-72260b220c8e', metadata={'offset': 34400}, page_content="Finally, we'll evaluate the model using accuracy, precision, recall, and F1 score. In the next lecture we'll explore decision trees and random forests in more detail.")]

In [17]:
llm = ChatOpenAI(
        model="openai/gpt-4o-mini",
        api_key=os.getenv("OPENROUTER_API_KEY"),
        base_url="https://openrouter.ai/api/v1"
)



In [ ]:
# from langchain_core.prompts import ChatPromptTemplate
# prompt = ChatPromptTemplate.from_template("""
# You are answering questions about a YouTube video.

# Rules:
# - Answer only using the transcript context.
# - Do not make up facts.
# - If the answer is not in the transcript, say you couldn't find it.
# - Cite the timestamp(s) from the context whenever possible.
# - Keep the answer concise unless the user asks for more detail.

# Transcript Context:
# {context}
                                          
# Timestamp:
# {timestamp}

# User Question:
# {question}

# Answer:
# """)

In [ ]:
# result

[Document(id='67e720d9-3c2b-458f-8bb0-72260b220c8e', metadata={'offset': 34400}, page_content="Finally, we'll evaluate the model using accuracy, precision, recall, and F1 score. In the next lecture we'll explore decision trees and random forests in more detail.")]

In [ ]:
# context_text = "\n\n".join(doc.page_content for doc in result)
# context_text

# # context_timestamp = []
# # for doc in result:
# #     context_timestamp.append(doc.metadata['offset'])

# # context_timestamp

[34400]

In [ ]:
# final_prompt = prompt.invoke({"context": context_text, "question": query})
# final_prompt

ChatPromptValue(messages=[HumanMessage(content="\nYou are answering questions about a YouTube video.\n\nRules:\n- Answer only using the transcript context.\n- Do not make up facts.\n- If the answer is not in the transcript, say you couldn't find it.\n- Cite the timestamp(s) from the context whenever possible.\n- Keep the answer concise unless the user asks for more detail.\n\nTranscript Context:\nFinally, we'll evaluate the model using accuracy, precision, recall, and F1 score. In the next lecture we'll explore decision trees and random forests in more detail.\n                                          \nTimestamp:\n[34400]\n\nUser Question:\nwhere did he talk about precision and recall\n\nAnswer:\n", additional_kwargs={}, response_metadata={})])

In [ ]:
# answer = llm.invoke(final_prompt)
# answer

AIMessage(content='He talked about precision and recall when evaluating the model, specifically mentioned at timestamp [34400].', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 134, 'total_tokens': 153, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'video_tokens': 0}, 'cost': 3.1185e-05, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 3.15e-05, 'upstream_inference_prompt_cost': 2.01e-05, 'upstream_inference_completions_cost': 1.14e-05}}, 'model_provider': 'openai', 'model_name': 'openai/gpt-4o-mini', 'system_fingerprint': 'fp_30570f0196', 'id': 'gen-1784740398-TozOhKA0CLS5pzZ9aBL2', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019f8ad1-4208-7aa0-9a2c-062008dc786f-0', tool_calls=[], invalid_t

In [ ]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
# from langchain_core.output_parsers import StrOutputParser, JsonOutputParser

In [50]:
def format_docs(retrieved_docs):
  context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text

def format_timestamp(retrieved_docs):
  context_timestamp = [doc.metadata["offset"] for doc in retrieved_docs]
  return context_timestamp


In [51]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough(),
    'timestamp' : retriever | RunnableLambda(format_timestamp)
})

In [56]:
# parser = StrOutputParser()
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

class output(BaseModel):

    content : str = Field(description='content to show')
    offset : int = Field(description='used for timestamp')

parser = PydanticOutputParser(pydantic_object=output)

In [57]:
from langchain_core.prompts import ChatPromptTemplate
prompt = ChatPromptTemplate.from_template("""
You are answering questions about a YouTube video.

Rules:
- Answer only using the transcript context.
- Do not make up facts.
- If the answer is not in the transcript, say you couldn't find it.
- Cite the timestamp(s) from the context whenever possible.
- Keep the answer concise unless the user asks for more detail.

Transcript Context:
{context}
                                          
Timestamp:
{timestamp}

User Question:
{question}

{format_instruction}
""",
partial_variables={'format_instruction' : parser.get_format_instructions()}
)

In [58]:
main_chain = parallel_chain | prompt | llm | parser

In [ ]:
result = main_chain.invoke("where did he talk about precision and recall")

'He talked about precision and recall while evaluating the model at timestamp [34400].'